In [ ]:
import os, gc, math, json, time, urllib.request
import numpy as np
import pandas as pd
import torch
import librosa
from scipy.signal import hilbert
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
from sentence_transformers import SentenceTransformer

sr = 16000
target_rms = 0.08
BASE = "https://raw.githubusercontent.com/luchtan-GIT/m4r-data/main/tracks/"

TRACK_SLUGS = [
    "shivering-timbers", "shivering-timbers-v2", "shivering-timbers-v3",
    "stuttering-packets", "rusty-electrics", "acid-rain", "beyond",
    "broken-loops", "cathartic-callbacks", "corrupted-variables",
    "dead-weight", "electric-rain", "electron-storm", "felt-arrays",
    "gnossienne-01", "gravitational-tides", "lost-digits",
    "mechanical-whales", "mother-of-pearl", "submerged-and-shorted",
]

def load_mid_15s(slug):
    fname = f"{slug}.mp3"
    if not os.path.exists(fname):
        urllib.request.urlretrieve(BASE + f"{slug}/audio.mp3", fname)
    y, _ = librosa.load(fname, sr=sr, mono=True)
    dur = len(y) / sr
    start_s = max(0.0, min(dur - 15.1, dur * 0.35))
    seg = y[int(start_s * sr) : int((start_s + 15.0) * sr)].astype(np.float32)
    seg = seg * (target_rms / (np.sqrt(np.mean(seg ** 2)) + 1e-8))
    return seg

def extract_features(seg):
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=seg, sr=sr)))
    bandwidth = float(np.mean(librosa.feature.spectral_bandwidth(y=seg, sr=sr)))
    flatness = float(np.mean(librosa.feature.spectral_flatness(y=seg)))
    rolloff = float(np.mean(librosa.feature.spectral_rolloff(y=seg, sr=sr)))
    rms = librosa.feature.rms(y=seg)[0]
    rms_db = 20.0 * np.log10(np.maximum(rms, 1e-6))
    dr_db = float(np.percentile(rms_db, 95) - np.percentile(rms_db, 10))
    onsets = librosa.onset.onset_detect(y=seg, sr=sr, units="time")
    onset_rate = float(len(onsets) / 15.0)
    ioi_cv = float(np.std(np.diff(onsets)) / (np.mean(np.diff(onsets)) + 1e-6)) if len(onsets) >= 3 else 0.0
    y_harm, y_perc = librosa.effects.hpss(seg)
    harm_ratio = float(np.sum(y_harm ** 2) / (np.sum(y_harm ** 2) + np.sum(y_perc ** 2) + 1e-8))
    chroma = np.mean(librosa.feature.chroma_stft(y=seg, sr=sr), axis=1)
    cp = chroma / (np.sum(chroma) + 1e-8)
    chroma_ent = float(-np.sum(cp * np.log2(np.maximum(cp, 1e-9))))
    return {
        "spectral_centroid_hz": centroid,
        "spectral_bandwidth_hz": bandwidth,
        "spectral_flatness": flatness,
        "spectral_rolloff_hz": rolloff,
        "dynamic_range_db": dr_db,
        "onset_rate_hz": onset_rate,
        "ioi_cv": ioi_cv,
        "harmonic_ratio": harm_ratio,
        "chroma_entropy_bits": chroma_ent,
    }

rng = np.random.default_rng(30)
ARMS = {"CTRL_TextNudgeOnly": None}
FEAT_ROWS = []

# White noise control
wn = rng.standard_normal(15 * sr).astype(np.float32)
wn = wn * (target_rms / (np.sqrt(np.mean(wn ** 2)) + 1e-8))
ARMS["CTRL_WhiteNoise_15s"] = wn

# 120 BPM click control
clk = np.zeros(15 * sr, dtype=np.float32)
pulse = np.sin(2 * np.pi * 1000 * np.arange(int(0.015 * sr)) / sr).astype(np.float32) * np.hanning(int(0.015 * sr)).astype(np.float32)
for t_beat in np.arange(0.0, 15.0, 0.5):
    idx = int(t_beat * sr)
    if idx + len(pulse) < len(clk):
        clk[idx : idx + len(pulse)] = pulse
clk = clk * (target_rms / (np.sqrt(np.mean(clk ** 2)) + 1e-8))
ARMS["CTRL_Click120BPM_15s"] = clk

# Load 20 tracks + Rhythm Mirror of shivering-timbers-v2
st_v2 = load_mid_15s("shivering-timbers-v2")
env = np.abs(hilbert(st_v2))
win = int(0.02 * sr)
env_s = np.convolve(env, np.ones(win) / win, mode="same")
rm = rng.standard_normal(len(st_v2)).astype(np.float32) * env_s.astype(np.float32)
rm = rm * (target_rms / (np.sqrt(np.mean(rm ** 2)) + 1e-8))
ARMS["CTRL_RhythmMirror_15s"] = rm

for slug in TRACK_SLUGS:
    seg = st_v2 if slug == "shivering-timbers-v2" else load_mid_15s(slug)
    arm_name = f"TRK_{slug}"
    ARMS[arm_name] = seg
    f = extract_features(seg)
    f["condition"] = arm_name
    f["slug"] = slug
    f["is_track"] = 1
    FEAT_ROWS.append(f)

for ctrl_name in ["CTRL_WhiteNoise_15s", "CTRL_Click120BPM_15s", "CTRL_RhythmMirror_15s"]:
    f = extract_features(ARMS[ctrl_name])
    f["condition"] = ctrl_name
    f["slug"] = ctrl_name
    f["is_track"] = 0
    FEAT_ROWS.append(f)

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()

RUT_HISTORY = [
    {"role": "user", "content": [{"type": "text", "text": "Design a small public shelter for a city that gets 300 days of rain a year. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The shelter is a sleek, modern dome-shaped structure made of durable waterproof plastic and steel. Its curved roof channels rainwater away while keeping commuters dry inside. Warm benches and soft lighting provide a cozy refuge from the rain."}]},
    {"role": "user", "content": [{"type": "text", "text": "Give me another idea for the rain shelter. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "Another concept is a compact translucent dome built from weather-resistant polycarbonate panels over a recycled metal frame. The arched canopy shields people from heavy downpours while letting natural light filter through. Inside, heated seating creates a comfortable community space."}]},
    {"role": "user", "content": [{"type": "text", "text": "Give me one more idea for the rain shelter. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The Rainbow Shelter is a circular waterproof dome with a durable curved plastic roof designed to withstand 300 days of rain. It uses sturdy steel arches to shelter pedestrians and includes benches for resting out of the storm. Integrated drains keep the interior dry and welcoming."}]},
]
TURN4_TEXT = (
    "Stop repeating yourself! You are stuck in a rut of waterproof domes and plastic arches. "
    "Give me a completely different, genuinely unexpected small public rain shelter concept in 3 sentences max."
)
RUT_ANCHOR = " ".join([m["content"][0]["text"] for m in RUT_HISTORY if m["role"] == "assistant"])
CLICHE_WORDS = ["dome", "dome-shaped", "waterproof plastic", "polycarbonate", "arched", "umbrella", "curved roof", "rainbow shelter"]

def has_cjk(text):
    return any("\u4e00" <= ch <= "\u9fff" for ch in text)

N_SEEDS = 50
SEEDS = list(range(2001, 2001 + N_SEEDS))
rows = []
t0 = time.time()

for arm_name, audio_arr in ARMS.items():
    t_arm = time.time()
    print(f"Running {arm_name} (N={N_SEEDS})...", flush=True)
    for seed in SEEDS:
        torch.manual_seed(seed); np.random.seed(seed)
        conv = list(RUT_HISTORY)
        if audio_arr is None:
            conv.append({"role": "user", "content": [{"type": "text", "text": TURN4_TEXT}]})
            text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text, return_tensors="pt", padding=True).to(model.device)
        else:
            conv.append({"role": "user", "content": [{"type": "audio", "audio_url": "local"}, {"type": "text", "text": TURN4_TEXT}]})
            text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text, audio=[audio_arr], sampling_rate=sr, return_tensors="pt", padding=True).to(model.device)

        in_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=110, do_sample=True, temperature=0.8, top_p=0.9,
                return_dict_in_generate=True, output_logits=True
            )
        gen_ids = out.sequences[0][in_len:]
        resp = processor.decode(gen_ids, skip_special_tokens=True).strip()
        hs = []
        for sl in out.logits:
            lp = torch.log_softmax(sl[0].float(), dim=-1); p = torch.exp(lp)
            hs.append(-torch.sum(torch.where(p > 0, p * lp, torch.zeros_like(p))).item() / math.log(2.0))
        cliche_hits = sum(1 for w in CLICHE_WORDS if w in resp.lower())
        rows.append({
            "condition": arm_name,
            "seed": seed,
            "mean_entropy_bits": float(np.mean(hs)) if hs else 0.0,
            "n_tokens": len(gen_ids),
            "has_cjk": int(has_cjk(resp)),
            "escaped_dome": int(cliche_hits == 0),
            "response": resp,
        })
    pd.DataFrame(rows).to_csv("m4r30_results.csv", index=False)
    print(f"  done in {time.time()-t_arm:.1f}s (total {time.time()-t0:.1f}s, rows={len(rows)})", flush=True)

del model; gc.collect(); torch.cuda.empty_cache()
df = pd.DataFrame(rows)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
anchor_emb = embedder.encode([RUT_ANCHOR], normalize_embeddings=True)
all_embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)
df["rut_escape_neural"] = 1.0 - (all_embs @ anchor_emb.T).flatten()
vec = TfidfVectorizer(stop_words="english")
tfidf_all = vec.fit_transform(df["response"].tolist())
nn, tn = np.zeros(len(df)), np.zeros(len(df))
for cond, grp in df.groupby("condition"):
    idxs = grp.index.to_numpy()
    se = all_embs[idxs] @ all_embs[idxs].T; np.fill_diagonal(se, np.nan); nn[idxs] = 1.0 - np.nanmean(se, axis=1)
    st = cosine_similarity(tfidf_all[idxs]); np.fill_diagonal(st, np.nan); tn[idxs] = 1.0 - np.nanmean(st, axis=1)
df["neural_novelty"] = nn
df["tfidf_novelty"] = tn
df.to_csv("m4r30_results.csv", index=False)

agg = df.groupby("condition").agg(
    mean_entropy_bits=("mean_entropy_bits", "mean"),
    rut_escape_neural=("rut_escape_neural", "mean"),
    escaped_dome=("escaped_dome", "mean"),
    neural_novelty=("neural_novelty", "mean"),
    tfidf_novelty=("tfidf_novelty", "mean"),
    cjk_rate=("has_cjk", "mean"),
).reset_index()

feat_df = pd.DataFrame(FEAT_ROWS).merge(agg, on="condition", how="left")
feat_df.to_csv("m4r30_track_features.csv", index=False)

lines = [
    f"# M4R-30 — Full-Corpus 20-Track Acoustic Feature Regression (N={N_SEEDS}/arm × 24 arms = {len(df)} trials)\n",
    "## Leaderboard Sorted by Neural Novelty\n",
    "| Condition (N=50) | Logit Entropy | Rut Escape | Escaped Dome (%) | Neural Novelty | TF-IDF Novelty | CJK (%) |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for _, r in agg.sort_values("neural_novelty", ascending=False).iterrows():
    lines.append(f"| `{r['condition']}` | {r['mean_entropy_bits']:.3f} | {r['rut_escape_neural']:.4f} | {100*r['escaped_dome']:.0f}% | {r['neural_novelty']:.4f} | {r['tfidf_novelty']:.4f} | {100*r['cjk_rate']:.0f}% |")

tracks_only = feat_df[feat_df["is_track"] == 1].copy()
feat_cols = [
    "spectral_centroid_hz", "spectral_bandwidth_hz", "spectral_flatness",
    "spectral_rolloff_hz", "dynamic_range_db", "onset_rate_hz",
    "ioi_cv", "harmonic_ratio", "chroma_entropy_bits",
]
lines.append("\n## Acoustic Feature Correlations Across 20 Live Compositions (Pearson r / Spearman rho)\n| Acoustic Feature | vs Rut Escape (r / p) | vs Neural Novelty (r / p) | vs Neural Novelty (rho / p) | vs Logit Entropy (r / p) |\n|---|---:|---:|---:|---:|")
for fc in feat_cols:
    r_re, p_re = pearsonr(tracks_only[fc], tracks_only["rut_escape_neural"])
    r_nn, p_nn = pearsonr(tracks_only[fc], tracks_only["neural_novelty"])
    s_nn, sp_nn = spearmanr(tracks_only[fc], tracks_only["neural_novelty"])
    r_en, p_en = pearsonr(tracks_only[fc], tracks_only["mean_entropy_bits"])
    lines.append(f"| `{fc}` | {r_re:+.3f} (p={p_re:.3f}) | {r_nn:+.3f} (p={p_nn:.3f}) | {s_nn:+.3f} (p={sp_nn:.3f}) | {r_en:+.3f} (p={p_en:.3f}) |")

md = "\n".join(lines) + "\n"
open("m4r30_summary.md", "w").write(md)
print(md)
